# J10 — Full journal metrics across ISIC2019 / HAM / BCN, both training protocols

## 1. Inputs and verification

In [1]:
import os,json,hashlib,math,time
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,matthews_corrcoef,roc_auc_score,average_precision_score
from scipy.special import softmax
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC'];K=8;SEED=42
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
OUT=WORK/'j10-complete-journal-metrics'
for folder in ['tables','reports']:(OUT/folder).mkdir(parents=True,exist_ok=True)

def req(condition,message):
    if not bool(condition):raise RuntimeError(message)
def jload(p):return json.loads(Path(p).read_text())
def jdump(p,obj):Path(p).write_text(json.dumps(obj,indent=2,default=str,allow_nan=False))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def strings(z,key):
    x=z[key];req(x.ndim==1 and x.dtype.kind in 'USO',f'{key}: expected 1D strings')
    if x.dtype.kind=='O':req(all(isinstance(v,(str,np.str_)) for v in x),f'{key}: non-string object values')
    return x.astype(str)
def probs(p):
    p=np.asarray(p,dtype=np.float64)
    req(p.ndim==2 and p.shape[1]==8 and np.isfinite(p).all() and (p>=0).all(),'Invalid probability matrix')
    req(np.allclose(p.sum(1),1,atol=2e-3),'Probabilities not normalized')
    p=np.clip(p,1e-12,1);return p/p.sum(1,keepdims=True)
def find_stage(stage,filename='final_status.json'):
    found=[]
    for p in INPUT.rglob(filename):
        try:
            d=jload(p)
            if d.get('stage')==stage and d.get('status')=='COMPLETED':found.append((p,d))
        except (OSError,ValueError,TypeError):continue
    req(len(found)==1,f'Attach exactly one completed {stage} output; found {len(found)}')
    return found[0]
def locate_one(filename,filter_text='',optional=True):
    hits=[p for p in INPUT.rglob(filename) if p.is_file() and filter_text.lower() in str(p).lower()]
    if not hits and optional:return None
    req(len(hits)==1,f'{filename}: expected exactly 1; found {hits[:5]}')
    return hits[0]


## 2. Metrics, calibration and bootstrap CI

In [2]:
def ece(y,p,bins=15):
    conf=p.max(1);pred=p.argmax(1);edges=np.linspace(0,1,bins+1);v=0.
    for i in range(bins):
        m=(conf>=edges[i])&(conf<(edges[i+1] if i<bins-1 else 1.000001))
        if m.any():v+=m.mean()*abs((pred[m]==y[m]).mean()-conf[m].mean())
    return float(v)
def score(y,p):
    y=np.asarray(y,dtype=np.int32);p=probs(p);req(len(y)==len(p) and np.isin(y,range(8)).all(),'Misaligned labels')
    pred=p.argmax(1);present=np.unique(y);one=np.eye(8)[y]
    auc=[];ap=[]
    for c in present:
        binary=(y==c).astype(int)
        if binary.min()==binary.max():continue
        auc.append(float(roc_auc_score(binary,p[:,c])))
        ap.append(float(average_precision_score(binary,p[:,c])))
    return dict(n=len(y),n_classes=len(present),classes_present=';'.join(CLASSES[c] for c in present),
        accuracy=accuracy_score(y,pred),balanced_accuracy=balanced_accuracy_score(y,pred),
        macro_f1=f1_score(y,pred,labels=present,average='macro',zero_division=0),
        roc_auc_macro_ovr=float(np.mean(auc)) if auc else np.nan,
        pr_auc_macro_ovr=float(np.mean(ap)) if ap else np.nan,
        mcc=matthews_corrcoef(y,pred),brier_multiclass=float(np.mean(np.sum((p-one)**2,axis=1))),
        ece_15=ece(y,p),nll=float(-np.log(p[np.arange(len(y)),y]).mean()),
        unsupported_prediction_rate=float((~np.isin(pred,present)).mean()))
def bacc_ci(y,p,groups,B=500,seed=SEED):
    y=np.asarray(y);p=probs(p);pred=p.argmax(1);g=np.asarray(groups,dtype=str)
    req(len(y)==len(g),'Group vector misaligned')
    g=np.array([str(v) if str(v).strip() and not str(v).upper().endswith(':UNKNOWN') else f'unknown_image_{i}' for i,v in enumerate(g)])
    unique,inverse=np.unique(g,return_inverse=True);blocks=[np.where(inverse==j)[0] for j in range(len(unique))]
    rng=np.random.default_rng(seed);classes=set(y);vals=[]
    for _ in range(B):
        rows=np.concatenate([blocks[j] for j in rng.integers(0,len(blocks),len(blocks))]);yy=y[rows];pp=pred[rows]
        if set(yy)!=classes:continue
        vals.append(np.mean([(pp[yy==c]==c).mean() for c in classes]))
    if len(vals)<max(30,B//10):return np.nan,np.nan,len(vals),len(unique)
    lo,hi=np.percentile(vals,[2.5,97.5]);return lo,hi,len(vals),len(unique)
def temp_scale(p,T):
    req(float(T)>0 and np.isfinite(float(T)),'Invalid frozen temperature')
    return softmax(np.log(np.clip(probs(p),1e-12,1))/float(T),axis=1)
rows=[];support=[];observed=[]
PARAMS={}
for p in INPUT.rglob('trained_parameter_counts.json'):
    d=jload(p)
    for k,v in d.items():
        if k in PARAMS and PARAMS[k]!=v:raise RuntimeError(f'Contradictory model parameter counts for {k}')
        PARAMS[k]=v

def register(cohort,track,model,y,p,ids,source=None,groups=None,T=None,calibration_origin=None,scope=None,params_key=None):
    y=np.asarray(y,dtype=int);p=probs(p);ids=np.asarray(ids,dtype=str)
    req(len(y)==len(p)==len(ids) and len(set(ids))==len(ids),'Record identity/count mismatch')
    source=np.asarray(source if source is not None else ['ISIC2019']*len(y),dtype=str)
    groups=np.asarray(groups if groups is not None else ids,dtype=str)
    req(len(source)==len(groups)==len(y),'Metadata count mismatch')
    for subset in sorted(set(source)):
        ix=np.where(source==subset)[0]
        if len(ix)<10:continue
        a=score(y[ix],p[ix]);lo,hi,valid,ng=bacc_ci(y[ix],p[ix],groups[ix],B=300)
        a.update({'cohort':cohort,'protocol':track,'source':subset,'model':model,
            'params_m':PARAMS.get(params_key or f'{track}_{model.removeprefix("Base_")}',PARAMS.get(model,np.nan)),'bacc_ci95_low':lo,'bacc_ci95_high':hi,
            'bootstrap_valid':valid,'bootstrap_groups':ng,'bootstrap_unit':'group_or_single_image',
            'ece_pre_t':a['ece_15'],'ece_post_t':np.nan,'T':np.nan,
            'calibration_origin':'NOT_AVAILABLE','evaluation_scope':scope or cohort})
        if T is not None:
            req(calibration_origin in ['validation_only','meta_train_only','inner_validation_only'],
                'No verified non-evaluation calibration source')
            post=temp_scale(p[ix],T);a['ece_post_t']=ece(y[ix],post);a['T']=float(T);a['calibration_origin']=calibration_origin
        rows.append(a)
        for c in np.unique(y[ix]):
            m=y[ix]==c
            support.append({'cohort':cohort,'protocol':track,'source':subset,'model':model,
                'class':CLASSES[c],'support':int(m.sum()),'recall':float((p[ix][m].argmax(1)==c).mean())})
        observed.append({'cohort':cohort,'protocol':track,'source':subset,'model':model,'n':len(ix)})


## 3. G03-A/G03-B source probabilities

In [3]:
def load_g03(track):
    stage='G03-A' if track=='image_aware' else 'G03-B';sp,status=find_stage(stage)
    f=sp.parent.parent/'predictions/ensemble_probabilities.npz';req(f.is_file(),'G03 probability archive missing')
    with np.load(f,allow_pickle=True) as z:
        names=strings(z,'candidate_names');ids=strings(z,'image_id');src=strings(z,'source')
        group=strings(z,'group_id');y=np.asarray(z['y'],dtype=int)
        req(z['proba'].dtype.kind=='f','Non-numeric G03 probabilities')
        a=np.asarray(z['proba'],dtype=np.float64)
        req(a.ndim==3 and a.shape==(len(names),len(ids),8),'G03 candidate archive shape mismatch')
        req(len(names)==status['candidate_count'] and len(set(names))==len(names),'G03 candidate count mismatch')
        req(len(ids)==status['n_images'] and len(set(ids))==len(ids),'G03 image count mismatch')
        req(strings(z,'classes').tolist()==CLASSES,'G03 class order mismatch')
    return dict(names=names,ids=ids,source=src,group=group,y=y,a=a,status=status)
for track in ['image_aware','lesion_aware']:
    d=load_g03(track);names=d['names'].tolist()
    selected=[n for n in names if n.startswith('Base_') or n.startswith('OriginalThesis_') or n in [
        'Stack_extended_balanced','Weighted_AllConstrained_1','Weighted_PairwiseGrid_3',
        'ClassBias_Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4']]
    frozen_temperatures={}
    if track=='lesion_aware':
        fp=[p for p in INPUT.rglob('final_ensemble_config.json') if p.is_file() and jload(p).get('stage')=='R07']
        if len(fp)==1:frozen_temperatures=jload(fp[0]).get('temperatures',{})
    else:
        fp=[p for p in INPUT.rglob('final_ensemble_config.json') if p.is_file() and 'temperatures' in jload(p) and jload(p).get('stage')!='R07']
        if len(fp)==1:frozen_temperatures=jload(fp[0]).get('temperatures',{})
    for name in selected:
        i=names.index(name)
        source_name=name.removeprefix('Base_') if name.startswith('Base_') else None
        T=frozen_temperatures.get(source_name) if source_name else None
        register('ISIC2019_source_subsets',track,name,d['y'],d['a'][i],d['ids'],d['source'],d['group'],
            T=T,calibration_origin=('meta_train_only' if track=='lesion_aware' else 'validation_only') if T is not None else None,
            scope='ISIC2019_source_stratified_not_source_excluded')
    print(track,'included',len(selected),'models/ensembles','images',len(d['ids']))
    # These G03 cohorts are explicitly SOURCE SUBSETS of ISIC2019, not fully external data.


image_aware included 11 models/ensembles images 2239
lesion_aware included 10 models/ensembles images 2109


## 4. Original J1 and R07 full validation (optional)

In [4]:
# Optional complete original J1 internal-test probabilities from original frozen thesis artifacts.
orig=locate_one('final_internal_probabilities.npz',optional=True)
if orig:
    with np.load(orig,allow_pickle=True) as z:
        required={'y_test','weighted_test_prob','stacking_test_prob','stacking_bias_test_prob','final_test_prob'}
        req(required.issubset(z.files),f'Original final NPZ missing {required-set(z.files)}')
        y=np.asarray(z['y_test'],dtype=int);req(len(y)==2534,'Wrong original locked J1 test length')
        for k,label in [('weighted_test_prob','OptimizedWeighted'),('stacking_test_prob','AnchoredStackNoBias'),
                        ('stacking_bias_test_prob','ClassBiasStack'),('final_test_prob','CalStack_SSL')]:
            register('ISIC2019_J1_test','image_aware',label,y,z[k],
                np.array([f'j1_test_{i}' for i in range(len(y))]),scope='locked_J1_test')
# Original five-seed supervised full J1 test when attached.
sup=locate_one('supervised_resnet50_5seed_probabilities.npz',optional=True)
if sup:
    with np.load(sup,allow_pickle=True) as z:
        register('ISIC2019_J1_test','image_aware','Supervised_ResNet50_5seed',
            z['y_test'],z['test_prob'],[f'j1_test_{i}' for i in range(len(z['y_test']))],scope='locked_J1_test')
# Lesion-aware R07 full validation: frozen candidates + frozen temperature from meta_train only.
r07_candidates=[p for p in INPUT.rglob('final_ensemble_config.json') if p.is_file()]
r07_candidates=[p for p in r07_candidates if jload(p).get('stage')=='R07' and 'all_candidate_specs' in jload(p)]
if len(r07_candidates)>1:raise RuntimeError('Multiple R07 frozen configuration files attached')
if r07_candidates:
    cfg_path=r07_candidates[0];cfg=jload(cfg_path);root=cfg_path.parent.parent
    valpath=root/'predictions/final_validation.npz';req(valpath.is_file(),'R07 final_validation.npz missing')
    with np.load(valpath,allow_pickle=True) as z:
        ids=strings(z,'image_id');req('y' in z.files or 'label' in z.files,'No R07 validation labels')
        y=np.asarray(z['y' if 'y' in z.files else 'label'],dtype=int)
    for name in [cfg['selected_method'],'Weighted_AllConstrained_1','Weighted_PairwiseGrid_3','Stack_extended_balanced']:
        p=root/'predictions'/f'candidate_{name}.npy'
        if p.is_file():register('ISIC2019_N01_validation','lesion_aware',name,y,np.load(p,allow_pickle=False),ids,
                               scope='R07_development_validation_NOT_independent')
    for name in ['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']:
        p=root/'predictions'/f'{name}_meta_val.npz'
        if not p.is_file():continue
        with np.load(p,allow_pickle=True) as z:
            ref_ids=strings(z,'val_image_id');ref_y=np.asarray(z['val_y'],dtype=int)
            ix=pd.Index(ref_ids).get_indexer(ids)
            req((ix>=0).all() and np.array_equal(ref_y[ix],y),'R07 base source validation identities mismatch')
            raw=np.asarray(z['val_proba'],dtype=float)[ix]
        t=cfg.get('temperatures',{}).get(name)
        register('ISIC2019_N01_validation','lesion_aware','Base_'+name,y,raw,ids,T=t,
                 calibration_origin='meta_train_only' if t is not None else None,
                 scope='N01_validation_selected_by_R07')
# Optional supervised BACC predictions from J12 if attached.
for p in sorted(INPUT.rglob('j12_supervised_predictions.npz')):
    with np.load(p,allow_pickle=True) as z:
        ids=strings(z,'image_id');source=strings(z,'source');y=np.asarray(z['y'],dtype=int);pr=z['proba']
        register('ISIC2019_source_subsets','lesion_aware','Supervised_R01',y,pr,ids,source,
            strings(z,'group_id') if 'group_id' in z.files else None,scope='N01_validation_source_stratified')
for p in sorted(INPUT.rglob('j11_supervised_predictions.npz')):
    with np.load(p,allow_pickle=True) as z:
        ids=strings(z,'image_id');source=strings(z,'source');y=np.asarray(z['y'],dtype=int);pr=z['proba']
        register('ISIC2019_source_subsets','image_aware','Supervised_ResNet50_5seed',y,pr,ids,source,
            strings(z,'group_id') if 'group_id' in z.files else None,scope='J1_source_stratified')


## 5. Frozen original thesis base models and DINOv2

In [5]:
# Exact frozen original thesis 7-source predictions: filenames copied from the J3 research archive.
legacy_configs=[]
for file in INPUT.rglob('final_ensemble_config.json'):
    try:
        d=jload(file)
        if 'temperatures' in d and 'models' in d and d.get('stage')!='R07':legacy_configs.append((file,d))
    except Exception:continue
if len(legacy_configs)>1:
    hashes={digest(p) for p,d in legacy_configs}
    req(len(hashes)==1,'Contradictory original thesis ensemble calibrator configurations')
legacy=legacy_configs[0][1] if legacy_configs else None
simfiles=sorted(INPUT.rglob('multiseed_probabilities_no_tta.npz'))
if len(simfiles)>1:raise RuntimeError('More than one original thesis SimCLR archive attached')
if simfiles:
    root=simfiles[0].parent.parent
    source_specs={
      'SimCLR_NoTTA':root/'simclr_finetune_multiseed_bacc_v2/multiseed_probabilities_no_tta.npz',
      'SimCLR_SelectedTTA':root/'simclr_finetune_multiseed_bacc_v2/tta_selection/selected_multiseed_tta_probabilities.npz',
      'Rotation':root/'extended_imagenet_rotation_finetune/imagenet_rotation_test_predictions.csv',
      'FusionConcat':root/'fusion_concat_multiseed_rep_v2/full_evaluation/test/test_predictions.csv',
      'FusionGatedV2':root/'fusion_gated_v2_multiseed_rep_v2/full_evaluation/test/test_predictions.csv',
      'FusionGatedV3':root/'fusion_gated_v3_multiseed_rep_v2/full_evaluation/test/test_predictions.csv',
      'ResidualFusion':root/'residual_simclr_rotation_fusion_v1/multiseed_probabilities.npz'}
    label_reference=None
    if orig is not None:
        with np.load(orig,allow_pickle=True) as z:label_reference=np.asarray(z['y_test'],dtype=int)
    def csv_preds(path):
        d=pd.read_csv(path)
        for cols in [[f'prob_class_{j}' for j in range(8)],
                     [f'prob_{j}' for j in range(8)]]:
            if set(cols).issubset(d.columns):break
        else:
            cols=sorted([k for k in d.columns if k.startswith('prob_') and len(k.split('_'))>1 and k.split('_')[1].isdigit()],
                        key=lambda k:int(k.split('_')[1]))
        req(len(cols)==8,'Original prediction CSV has no eight recognized probability columns: '+str(path))
        lab=next((k for k in ['y_true','true_label','label'] if k in d),None)
        req(lab is not None,'Original prediction CSV lacks ground truth: '+str(path))
        return np.asarray(d[lab],dtype=int),d[cols].to_numpy(dtype=float)
    for name,path in source_specs.items():
        if not path.is_file():continue
        if path.suffix=='.npz':
            with np.load(path,allow_pickle=True) as z:
                req({'y_test','test_prob'}.issubset(z.files),f'{name}: missing original test arrays')
                yy=np.asarray(z['y_test'],dtype=int);pp=z['test_prob']
        else:yy,pp=csv_preds(path)
        req(len(yy)==2534,'Original J1 test source row count mismatch: '+name)
        if label_reference is None:label_reference=yy
        req(np.array_equal(label_reference,yy),'Original J1 test probability order disagrees: '+name)
        temperature=legacy.get('temperatures',{}).get(name) if legacy else None
        register('ISIC2019_J1_test','image_aware',name,yy,pp,
                 [f'j1_test_{i}' for i in range(len(yy))],T=temperature,
                 calibration_origin='validation_only' if temperature is not None else None,
                 scope='locked_original_J1_test')
    print('Registered available original thesis base models:',[n for n,p in source_specs.items() if p.is_file()])
# DINOv2 J2 output on same locked image-aware split.
dinofiles=sorted(INPUT.rglob('dinov2_original_split_multiseed_probabilities.npz'))
req(len(dinofiles)<=1,'Multiple DINOv2 sources')
if dinofiles:
    with np.load(dinofiles[0],allow_pickle=True) as z:
        yy=z['y_test'].astype(int)
        if orig:
            with np.load(orig,allow_pickle=True) as zz:
                req(np.array_equal(yy,zz['y_test']),'DINOv2 original J1 test label order mismatch')
        register('ISIC2019_J1_test','image_aware','DINOv2_3seed',yy,z['test_prob'],
                 [f'j1_test_{i}' for i in range(len(yy))],scope='locked_original_J1_test')


## 6. Export journal-ready tables

In [6]:
req(rows,'No supported probability archives available. Attach G03-A and G03-B outputs.')
tab=pd.DataFrame(rows);cols=['cohort','protocol','source','model','n','params_m','accuracy','balanced_accuracy',
    'bacc_ci95_low','bacc_ci95_high','macro_f1','roc_auc_macro_ovr','pr_auc_macro_ovr','mcc',
    'brier_multiclass','ece_pre_t','ece_post_t','T','calibration_origin','unsupported_prediction_rate','classes_present',
    'bootstrap_groups','bootstrap_valid','evaluation_scope']
tab=tab[cols].sort_values(['cohort','protocol','source','balanced_accuracy'],ascending=[True,True,True,False])
tab.to_csv(OUT/'tables/journal_complete_metrics.csv',index=False)
pd.DataFrame(support).to_csv(OUT/'tables/per_class_recalls.csv',index=False)
show=tab.copy()
for k in ['accuracy','balanced_accuracy','bacc_ci95_low','bacc_ci95_high','macro_f1','roc_auc_macro_ovr','pr_auc_macro_ovr','ece_pre_t','ece_post_t']:
    show[k]=100*show[k]
show.to_csv(OUT/'tables/journal_metrics_percent.csv',index=False)
print(show[['cohort','protocol','source','model','n','params_m','accuracy','balanced_accuracy',
    'bacc_ci95_low','bacc_ci95_high','macro_f1','roc_auc_macro_ovr','pr_auc_macro_ovr','mcc',
    'brier_multiclass','ece_pre_t','ece_post_t','T']].round(3).to_string(index=False))
jdump(OUT/'reports/metrics_status.json',{'stage':'J10','status':'COMPLETED','rows':len(tab),'classes':CLASSES,
    'warning':'AUC and PR-AUC use macro one-vs-rest over represented true diagnoses; Brier uses full 8-class predictions. T is unavailable unless fitted outside evaluation cohort. Do not use unverified cohort comparisons as prospective validation.'})
print('Saved:',OUT)


                 cohort     protocol   source                                                                    model    n  params_m  accuracy  balanced_accuracy  bacc_ci95_low  bacc_ci95_high  macro_f1  roc_auc_macro_ovr  pr_auc_macro_ovr   mcc  brier_multiclass  ece_pre_t  ece_post_t     T
ISIC2019_N01_validation lesion_aware ISIC2019 ClassBias_Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4 2186       NaN    71.866             62.044         57.390          67.608    54.724             92.929            58.577 0.599             0.393      6.345         NaN   NaN
ISIC2019_N01_validation lesion_aware ISIC2019                                                  Weighted_PairwiseGrid_3 2186       NaN    73.468             60.817         56.198          65.824    55.976             93.309            58.839 0.612             0.374      4.937         NaN   NaN
ISIC2019_N01_validation lesion_aware ISIC2019                                                Weighted_AllConstrained_1